# 🎯 Module 13 · 08 — Sentiment Analysis

> **Stage:** Machine Learning → NLP  
> **Level:** Intermediate  
> **Module:** 13 — NLP Basics / 08 — Sentiment Analysis  
> **Date:** 1 February 2026  
> **Previous:** 07 — Text Classification  
> **Next:** 09 — NLP Evaluation

---

**Sentiment Analysis** identifies the **emotional or opinion-based meaning** of text.

Most common task: classify text as **Positive**, **Negative**, or **Neutral**.

In this notebook we'll cover:

- What sentiment analysis is
- Sentiment as **classification**
- The classical pipeline
- **Negation** handling
- **N-grams** for sentiment
- **Text representation**
- **Classical models** (NB, LR, LinearSVC)
- **Pipeline** usage
- **Evaluation**, confusion matrix, **class imbalance**
- **Feature engineering**
- **Emojis** and **social-media text**
- Sentiment vs **emotion**
- **Aspect-based** sentiment
- **Error analysis**
- **Saving** and **deploying** the model

## 📚 Learning Objectives

By the end of this notebook you should be able to:

- Explain what sentiment analysis is
- Frame sentiment as **binary / multiclass classification**
- Handle **negation** in preprocessing
- Use **n-grams** to capture phrases like `"not good"`
- Build a sentiment classifier with **TF-IDF + Logistic Regression**
- Evaluate with precision, recall, F1, and **confusion matrix**
- Handle **class imbalance** with `class_weight='balanced'`
- Use **word** and **character** features
- Reason about **emojis** and **social-media text**
- Distinguish **sentiment** from **emotion**
- Explain **aspect-based** sentiment
- Perform **error analysis**
- **Save** and **deploy** a sentiment pipeline

---
## ⚙️ 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
)

np.random.seed(42)
print('✅ Setup ready.')

---
## 🧠 1. What Is Sentiment Analysis?

Automatically detecting the **sentiment or opinion** expressed in text.

```text
Input Text
     ↓
Preprocessing
     ↓
Text Representation
     ↓
ML Model
     ↓
Sentiment
```

### Examples

```text
"I really love this product"     →  Positive
"The service was terrible"       →  Negative
"The package arrived today"      →  Neutral
```

> The model doesn't **understand** like a human — it learns **patterns** from labeled examples.

---
## 🏷️ 2. Sentiment as Classification

### Binary

```text
Positive / Negative
```

| Text | Label |
| --- | --- |
| I love this movie | Positive |
| This movie is boring | Negative |

### Three-class

```text
Positive / Negative / Neutral
```

| Text | Label |
| --- | --- |
| Amazing experience | Positive |
| Terrible experience | Negative |
| The package arrived | Neutral |

### Multi-class

```text
Very Negative
Negative
Neutral
Positive
Very Positive
```

The number of classes depends on the **dataset** and the **business need**.

---
## 🔁 3. Sentiment Analysis Pipeline

```text
Raw Text
   ↓
Text Cleaning
   ↓
Tokenization
   ↓
Normalization
   ↓
TF-IDF / BoW
   ↓
Train ML Model
   ↓
Evaluate
   ↓
Predict Sentiment
```

Example:

```text
"I absolutely LOVE this phone!!!"
              ↓
Cleaning
              ↓
"absolutely love phone"
              ↓
TF-IDF
              ↓
Logistic Regression
              ↓
Positive
```

---
## 📊 4. Dataset

A sentiment dataset contains:

```text
text → sentiment label
```

Split:

```text
Training Set   → learn
Validation Set → tune / select
Test Set       → final generalization
```

In [ ]:
# Small synthetic sentiment dataset for a full demo
positive = [
    "I love this product", "Amazing quality", "Absolutely wonderful",
    "Best purchase ever", "Highly recommend", "Fantastic experience",
    "Great service", "Excellent quality", "Very happy with this",
    "Exceeded expectations", "Really enjoyed it", "Superb product",
]
negative = [
    "Terrible quality", "Absolutely awful", "Worst purchase ever",
    "Do not recommend", "Horrible experience", "Bad service",
    "Very disappointed", "Waste of money", "Poor quality",
    "Below expectations", "Really disliked it", "I hate it",
]

texts  = positive + negative
labels = ['positive'] * len(positive) + ['negative'] * len(negative)

df = pd.DataFrame({'text': texts, 'sentiment': labels})
print('Dataset shape:', df.shape)
print(df['sentiment'].value_counts())

---
## 🧹 5. Text Preprocessing for Sentiment

Common operations:

- Lowercasing
- Removing HTML
- Cleaning URLs
- Handling punctuation
- Whitespace normalization
- Tokenization
- Stop-word handling
- Stemming / Lemmatization
- Handling emojis
- Handling repeated characters

### ⚠️ But be careful — some information matters!

```text
"good"
"not good"
```

have **completely different meanings**.

> Blind aggressive preprocessing often **hurts** sentiment models.

---
## 🚫 6. Negation Handling

**Negation** is one of the **most important** problems in sentiment analysis.

```text
"This phone is good"
"This phone is not good"
```

The word **not** changes the sentiment entirely.

### How to handle negation

- Keep important negation words (`not`, `never`, `no`)
- Use **n-grams** to preserve `"not good"`
- **Negation-aware** preprocessing
- Feature engineering
- More advanced models

> ⚠️ Do **not** automatically remove every stop word when building a sentiment model.

In [ ]:
# Show why negation matters
sentences = [
    "this phone is good",
    "this phone is not good",
    "i like this product",
    "i do not like this product",
]

STOP = {'this', 'is', 'i', 'do', 'a', 'the'}

print(f'{"Original":<35} {"Naive filter (no not)":<30} {"Negation-aware":<30}')
print('-' * 96)
for s in sentences:
    tokens = s.split()
    naive  = ' '.join(t for t in tokens if t not in STOP)
    keep_not = {'not', 'no', 'never'}
    aware = ' '.join(t for t in tokens if t not in (STOP - keep_not))
    print(f'{s:<35} {naive:<30} {aware:<30}')

---
## 🔗 7. N-Grams for Sentiment

Single words are often not enough.

```text
"not good"
```

- Unigrams: `not`, `good`
- Bigrams:  `not good`  ✅ captures the sentiment

```python
TfidfVectorizer(ngram_range=(1, 2))
```

### Useful n-grams for sentiment

```text
very good
not good
really bad
highly recommend
waste of
```

---
## 🔢 8. Text Representation

Models need numbers:

### Bag of Words
```text
word → frequency
```

### TF-IDF
```text
word → importance
```

### Word Embeddings
```text
word → dense vector
```

> For **classical** sentiment analysis, **TF-IDF + linear models** is a strong, practical baseline.

---
## 🤖 9. Machine Learning Models

### Naive Bayes
```text
TF-IDF / BoW  →  Naive Bayes  →  Sentiment
```
Simple, fast, strong baseline for text.

### Logistic Regression
```text
TF-IDF  →  Logistic Regression  →  Positive / Negative
```
Strong, interpretable baseline.

### Linear SVM
```text
TF-IDF  →  Linear SVM  →  Sentiment
```
Often best on **sparse high-dimensional** text features.

> Pick the winner on **validation**, not by guessing.

---
## 🧪 10. Complete Classical Example

```python
model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=2)),
    ('classifier', LogisticRegression(max_iter=1000)),
])
model.fit(X_train, y_train)
predictions = model.predict(X_test)
```

In [ ]:
# Train / test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['sentiment'],
    test_size=0.25, random_state=42, stratify=df['sentiment'],
)

# Full pipeline
model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=1)),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42)),
])

model.fit(X_train, y_train)
predictions = model.predict(X_test)

print('Test accuracy:', round(accuracy_score(y_test, predictions), 3))
print()
print(classification_report(y_test, predictions, digits=3))

In [ ]:
# Predict on new text
new_texts = [
    "I absolutely love this product",
    "This service is terrible",
    "The package arrived today",
]

print('Predictions:')
for t, p in zip(new_texts, model.predict(new_texts)):
    print(f'  {t!r:<45} -> {p}')

---
## 🏗️ 11. Why Use a Pipeline?

**Without** a pipeline: you must manually ensure the same TF-IDF transformation is applied during **train** and **inference** — easy to get wrong.

**With** a pipeline:

```python
Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('model', LogisticRegression()),
])
```

- Reusable
- Safer
- **Prevents data leakage**
- One object to **save**, **load**, and **deploy**

---
## 📊 12. Evaluation

Accuracy alone is **not enough**.

Important metrics:

- **Accuracy**
- **Precision**
- **Recall**
- **F1-score**
- **Confusion matrix**

```python
from sklearn.metrics import classification_report
print(classification_report(y_test, predictions))
```

> On **imbalanced** datasets, **F1 / precision / recall** are more informative than accuracy.

---
## 🔢 13. Confusion Matrix

```text
                 Predicted
              Positive Negative
Actual Positive    TP      FN
Actual Negative    FP      TN
```

Shows **where** the model makes mistakes.

For example, does it struggle more with **positive** or **negative** examples?

In [ ]:
# Confusion matrix heatmap
labels_order = model.classes_
cm = confusion_matrix(y_test, predictions, labels=labels_order)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(labels_order)))
ax.set_yticks(range(len(labels_order)))
ax.set_xticklabels(labels_order)
ax.set_yticklabels(labels_order)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
for i in range(len(labels_order)):
    for j in range(len(labels_order)):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=13)
plt.colorbar(im)
plt.title('Confusion Matrix')
plt.tight_layout(); plt.show()

---
## ⚖️ 14. Class Imbalance

If a dataset is:

```text
Positive 90%
Negative 10%
```

predicting positive often gives high **accuracy** but useless **negative recall**.

### Solutions

- **Class weights**
- Better sampling
- More balanced data
- Better metrics (F1, PR-AUC)

```python
LogisticRegression(class_weight='balanced')
```

In [ ]:
# Simulate class imbalance and compare default vs balanced
imb_texts  = positive * 9 + negative * 1   # 90% / 10%
imb_labels = ['positive'] * (len(positive) * 9) + ['negative'] * len(negative)

Xtr_i, Xte_i, ytr_i, yte_i = train_test_split(
    imb_texts, imb_labels,
    test_size=0.25, random_state=42, stratify=imb_labels,
)

for cw in [None, 'balanced']:
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2))),
        ('clf', LogisticRegression(max_iter=1000, class_weight=cw, random_state=42)),
    ])
    pipe.fit(Xtr_i, ytr_i)
    p = pipe.predict(Xte_i)
    tag = 'default  ' if cw is None else 'balanced '
    print(f'{tag} acc={accuracy_score(yte_i, p):.3f}  '
          f'f1(macro)={f1_score(yte_i, p, average="macro"):.3f}  '
          f'recall(negative)={recall_score(yte_i, p, pos_label="negative"):.3f}')

---
## 🛠️ 15. Feature Engineering

### Word features
```text
good, bad, excellent, terrible
```

### Word n-grams
```text
very good, not good, really bad
```

### Character n-grams
Useful for:

- Misspellings
- Slang
- Repeated characters
- Social media text

Example:
```text
goooood
```

Character features can capture patterns that word-level features miss.

In [ ]:
# Compare word vs character features on noisy sentiment text
noisy_texts = [
    "I loveee this product!!!",
    "soooo good, best ever!",
    "absolutely terrible",
    "very baaad quality",
    "amazinggg!!",
    "worst thing everrr",
]
noisy_labels = ['positive', 'positive', 'negative', 'negative', 'positive', 'negative']

configs = {
    'word (1,2)':    TfidfVectorizer(analyzer='word', ngram_range=(1, 2)),
    'char_wb (3,5)': TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5)),
    'union':         None,   # placeholder
}

for name, vec in [c for c in configs.items() if c[1] is not None]:
    pipe = Pipeline([
        ('vec', vec),
        ('clf', LogisticRegression(max_iter=1000, random_state=42)),
    ])
    # Use LOO-ish style: train and test on the same small set (illustrative only)
    pipe.fit(noisy_texts, noisy_labels)
    preds = pipe.predict(noisy_texts)
    print(f'{name:<16} train-acc={accuracy_score(noisy_labels, preds):.3f}')

---
## 😍 16. Emojis and Sentiment

Emojis can contain **strong sentiment** information:

```text
"I love this 😍"
"I hate this 😡"
```

### Options

- **Keep** emojis
- **Convert** them to text descriptions
- **Map** them to sentiment features
- **Remove** them if irrelevant

> Removing emojis **blindly** can hurt sentiment models.

---
## 📱 17. Social Media Sentiment

```text
"This phone is soooo good 😂🔥"
```

### Challenges

- Slang
- Hashtags
- Mentions
- Emojis
- Misspellings
- Repeated characters
- Short sentences
- **Code-switching** (mixed languages)

Example (Roman Urdu + English):
```text
"ye phone bohat acha hai 🔥"
```

May need **specialized preprocessing** and datasets.

---
## 🎭 18. Sentiment vs Emotion

| Sentiment | Emotion |
| --- | --- |
| Positive / Negative / Neutral | Joy, Anger, Sadness, Fear, Surprise, Disgust |
| Polarity | Specific category |

Example:

```text
"I am extremely angry about this service"
```

```text
Sentiment → Negative
Emotion   → Anger
```

---
## 🔬 19. Aspect-Based Sentiment Analysis

A review can express **different opinions about different aspects**:

```text
"The camera is excellent but the battery is terrible."
```

Overall sentiment loses information.

### Aspect-based analysis

```text
Camera  → Positive
Battery → Negative
```

Useful for:

- Product reviews
- Restaurant reviews
- Customer feedback
- Service analysis

More complex than simple positive/negative classification.

---
## 🔍 20. Error Analysis

Inspect **wrong predictions** — metrics alone hide the real story.

### Common error categories

- **Negation**
- **Sarcasm**
- Slang
- Spelling errors
- **Mixed languages**
- Ambiguous text
- Long text
- Domain-specific language
- Incorrect labels

### Example

```text
"The battery is not bad"
Actual    → Positive
Predicted → Negative  ← negation error
```

```text
"Yeah, amazing... another broken update."
Predicted → Positive  ← sarcasm error
```

In [ ]:
# Show wrong predictions on the small test set
preds = model.predict(X_test)
wrong = [(t, a, p) for t, a, p in zip(X_test, y_test, preds) if a != p]

print(f'Test samples    : {len(X_test)}')
print(f'Misclassified   : {len(wrong)}')
print()
if wrong:
    print('Examples:')
    for text, actual, pred in wrong[:5]:
        print(f'  {text!r}')
        print(f'    actual={actual}  predicted={pred}')
else:
    print('No misclassifications on this tiny test set (uncommon but possible).')

---
## ⚠️ 21. Common Challenges

### Sarcasm
```text
"Wow, what an amazing way to waste my time."
```

### Context
```text
"This is sick!"     ← positive slang
```

### Negation
```text
"not bad"
```

### Mixed sentiment
```text
"Great camera but terrible battery."
```

### Domain differences
```text
"unpredictable"  — good in films, bad in surgery
```

### Code-switching
```text
"phone bohat acha hai but battery weak hai"
```

These cases are why **keyword-based** approaches have hard limits.

---
## 📏 22. Baseline First

```text
1. Prepare dataset
       ↓
2. Clean text
       ↓
3. TF-IDF
       ↓
4. Logistic Regression
       ↓
5. Evaluate
       ↓
6. Error analysis
       ↓
7. Try improvements
```

> Don't start with a complicated system. **Measure improvement over a simple baseline.**

---
## 🎛️ 23. Hyperparameter Tuning

### TF-IDF
```text
ngram_range
min_df
max_df
max_features
sublinear_tf
```

### Logistic Regression
```text
C
class_weight
```

### Linear SVM
```text
C
class_weight
```

Use `GridSearchCV` or `RandomizedSearchCV`.

In [ ]:
# Small grid search
pipe = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression(max_iter=1000, random_state=42)),
])

params = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__min_df':      [1, 2],
    'clf__C':             [0.1, 1, 10],
}

grid = GridSearchCV(pipe, params, cv=3, scoring='f1_macro', n_jobs=-1)
grid.fit(X_train, y_train)

print('Best params:', grid.best_params_)
print(f'Best CV F1 : {grid.best_score_:.4f}')
print(f'Test F1    : {f1_score(y_test, grid.predict(X_test), average="macro"):.4f}')

---
## 💾 24. Saving the Model

```python
import joblib
joblib.dump(model, 'sentiment_pipeline.pkl')

# later
model = joblib.load('sentiment_pipeline.pkl')
```

The complete pipeline can then accept raw text and produce predictions — no manual vectorization needed.

In [ ]:
# Save & load the model (comment out to avoid writing files)
import joblib
from pathlib import Path

# out_path = Path('sentiment_pipeline.pkl')
# joblib.dump(model, out_path)
# loaded = joblib.load(out_path)
# print('Loaded prediction:', loaded.predict(['absolutely amazing product'])[0])

print('Uncomment the lines above to save / load the pipeline.')

---
## 🚀 25. Production Architecture

```text
User Text
    ↓
FastAPI
    ↓
Saved NLP Pipeline
    ↓
TF-IDF
    ↓
Classifier
    ↓
Sentiment
    ↓
JSON Response
```

Example response:

```json
{
  "text": "I love this product",
  "sentiment": "positive"
}
```

Package with **Docker** for deployment.

---
## 🌍 26. Real-World Applications

### Customer Feedback
```text
Customer reviews → Sentiment → Business insights
```

### Product Reviews
```text
Reviews → Positive / Negative → Product analysis
```

### Support Tickets
```text
Support messages → Sentiment → Prioritization
```

### Social Media
```text
Posts → Sentiment → Opinion monitoring
```

---
## 🔄 27. Classical vs Modern Sentiment Analysis

### Classical

```text
Text → Preprocessing → TF-IDF / BoW → ML Model → Sentiment
```

### Modern

```text
Text → Transformer Model → Contextual Representation → Classification
```

### Why classical still matters

- Text preprocessing
- Feature engineering
- Sparse representations
- Classification
- Evaluation
- Error analysis
- Production pipelines

These concepts are **foundational** for modern NLP.

---
## 🧠 28. Mental Model

```text
TEXT
 ↓
CLEAN
 ↓
TOKENIZE
 ↓
HANDLE NEGATION
 ↓
REPRESENT (TF-IDF / Embeddings)
 ↓
CLASSIFY (LR / NB / SVM)
 ↓
EVALUATE
 ↓
ERROR ANALYSIS
 ↓
IMPROVE
 ↓
DEPLOY
```

### 🧭 Golden Rule

> **Good sentiment analysis is not just choosing a classifier. It depends on the quality of the text preprocessing, representation, dataset, evaluation, and error analysis.**

---
## 🏋️ 29. Hands-On Exercises

### Exercise 1 — Binary sentiment
Build a classifier on this small corpus:
```python
texts = [
    "I love this", "Best ever", "Amazing product",
    "Terrible", "Worst ever", "Hated it",
]
labels = ['pos', 'pos', 'pos', 'neg', 'neg', 'neg']
```
Report accuracy, precision, recall, F1.

### Exercise 2 — Negation
Add these two sentences to a sentiment dataset:
- `"This is good"`
- `"This is not good"`

How does a classifier with **unigrams only** handle them vs one with **bigrams**?

### Exercise 3 — Emoji impact
Compare two versions of the same dataset:
1. Emojis removed
2. Emojis converted to text (`😍 → heart_eyes`)

Which gives better F1?

### Exercise 4 — Class imbalance
Build a 90/10 imbalanced sentiment dataset.  
Compare `class_weight=None` vs `class_weight='balanced'`.  
Focus on the **minority class** recall.

### Exercise 5 — Aspect-based
Write a rule-based aspect extractor for:
```text
"The camera is excellent but the battery is terrible."
```
Identify each aspect and its sentiment.

### Exercise 6 — Error analysis
Take a trained model's misclassified examples.  
Group them into categories (negation, sarcasm, slang, etc.).  
Which category dominates? What could you try next?

---
## 🎤 30. Interview Questions

1. **What is sentiment analysis?**  
   Detecting the emotional polarity of text — positive, negative, or neutral.

2. **Is it supervised?**  
   Typically supervised — trained on labeled examples.

3. **Binary vs multiclass sentiment?**  
   Binary: pos/neg. Multiclass: pos/neg/neutral, or finer.

4. **Why is negation important?**  
   `not good` ≠ `good` — removing negation flips meaning.

5. **How do n-grams help?**  
   They capture phrases like `"not good"` that unigrams miss.

6. **Best classical baseline?**  
   TF-IDF + Logistic Regression (or LinearSVC).

7. **Why use a Pipeline?**  
   Consistent transformation, reusable, prevents leakage, single artifact.

8. **Why not rely on accuracy?**  
   Imbalanced classes can produce high accuracy with useless minority performance.

9. **How do you handle class imbalance?**  
   `class_weight='balanced'`, resampling, better metrics, more data.

10. **Word vs character features?**  
    Word features capture meaning; char features handle typos, slang, and noisy text.

11. **Should you remove emojis?**  
    Not blindly — they often carry sentiment signal.

12. **Sentiment vs emotion?**  
    Sentiment = polarity. Emotion = specific category (joy, anger, etc.).

13. **What is aspect-based sentiment?**  
    Assigning sentiment per **aspect** of a text (e.g., camera vs battery).

14. **Why is sentiment analysis hard?**  
    Sarcasm, negation, slang, mixed sentiment, domain-specific language, code-switching.

15. **How do you deploy a sentiment model?**  
    Save the whole Pipeline with `joblib`, expose via FastAPI, containerize with Docker.

---
## 🏁 31. Key Takeaways

### You now understand

- What sentiment analysis is
- Sentiment as classification
- Binary and multiclass sentiment
- Preprocessing for sentiment
- Why **negation matters**
- **N-grams** for sentiment
- TF-IDF for sentiment classification
- Logistic Regression / NB / LinearSVC
- Evaluation metrics
- **Confusion matrix**
- **Class imbalance**
- Feature engineering
- **Character n-grams**
- **Emojis** and social-media text
- **Aspect-based** sentiment
- Sentiment vs **emotion**
- **Error analysis**
- Model pipelines
- Hyperparameter tuning
- **Saving** a trained pipeline
- **Basic production deployment**

### The full pipeline

```text
Text → Clean → TF-IDF → Classifier → Evaluate → Deploy
```

---

### 🔗 What's Next?

**09 — NLP Evaluation**

We'll focus specifically on **evaluating NLP systems** — accuracy, precision, recall, F1, confusion matrix, ROC-AUC, cross-validation, class imbalance, error analysis, and NLP-specific considerations.